<a href="https://colab.research.google.com/github/remix0091/Big-Data/blob/main/L1_%D0%92%D0%B2%D0%B5%D0%B4%D0%B5%D0%BD%D0%B8%D0%B5_%D0%B2_Mapreduce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Введение в MapReduce модель на Python


In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [ ]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [ ]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [ ]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [ ]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [ ]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [ ]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [ ]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [ ]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [ ]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [ ]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.3614589596337598)),
 (1, np.float64(1.3614589596337598)),
 (2, np.float64(1.3614589596337598)),
 (3, np.float64(1.3614589596337598)),
 (4, np.float64(1.3614589596337598))]

## Inverted index

In [ ]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [ ]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [ ]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('it', 18)]),
 (1, [('a', 2), ('is', 18), ('what', 10)])]

## TeraSort

In [ ]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.036812798499743726)),
   (None, np.float64(0.04149296681948378)),
   (None, np.float64(0.09304452338205227)),
   (None, np.float64(0.12307404267451494)),
   (None, np.float64(0.1848168751322503)),
   (None, np.float64(0.19896820808707072)),
   (None, np.float64(0.3110496340366342)),
   (None, np.float64(0.3118322546056941)),
   (None, np.float64(0.3346419921374876)),
   (None, np.float64(0.46446168569479906)),
   (None, np.float64(0.46457513594334354)),
   (None, np.float64(0.4713877341406303)),
   (None, np.float64(0.4781366591873696)),
   (None, np.float64(0.48245963523068014)),
   (None, np.float64(0.49447116347333464))]),
 (1,
  [(None, np.float64(0.5450006063931636)),
   (None, np.float64(0.5506307876308683)),
   (None, np.float64(0.5701436548893117)),
   (None, np.float64(0.5952182075612557)),
   (None, np.float64(0.6241230544376554)),
   (None, np.float64(0.6408304930778682)),
   (None, np.float64(0.6694289405352155)),
   (None, np.float64(0.713131838

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [ ]:
input_collection = [4, 11, 7, 25, 3, 18, 25, 2]

def RECORDREADER():
  for i, value in enumerate(input_collection):
    yield (i, value)

def MAP(k1, v1):
  yield ('max', v1)

def REDUCE(key, values:Iterator[int]):
  max_value = None
  for value in values:
    if (max_value is None) or (value > max_value):
      max_value = value
  yield (key, max_value)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('max', 25)]

### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [ ]:
def RECORDREADER():
  for i, value in enumerate(input_collection):
    yield (i, value)

def MAP(k1, v1):
  yield ('avg', (v1, 1))

def REDUCE(key, values:Iterator[tuple[int, int]]):
  total_sum = 0
  total_count = 0
  for value, count in values:
    total_sum += value
    total_count += count
  yield (key, total_sum / total_count)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('avg', 11.875)]

### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [ ]:
def groupbykey_sorting(iterable):
  data = sorted(iterable, key=lambda x: x[0])

  current_key = None
  current_values = None

  for (k2, v2) in data:
    if current_key != k2:
      if current_key is not None:
        yield (current_key, current_values)
      current_key = k2
      current_values = [v2]
    else:
      current_values.append(v2)

  if current_key is not None:
    yield (current_key, current_values)

example = [
    ('b', 2),
    ('a', 1),
    ('b', 3),
    ('c', 7),
    ('a', 5),
    ('c', 8),
    ('b', 4)
]

list(groupbykey_sorting(example))

[('a', [1, 5]), ('b', [2, 3, 4]), ('c', [7, 8])]

### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [ ]:
input_collection = [4, 11, 7, 25, 3, 18, 25, 2, 11, 7, 4, 30]

def MAP(k1, v1):
  yield (v1, 1)

def REDUCE(key, values):
  yield key

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[4, 11, 7, 25, 3, 18, 2, 30]

#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [ ]:
class Row(NamedTuple):
  id: int
  name: str
  age: int
  city: str

input_collection = [
    Row(0, 'Ann', 19, 'Moscow'),
    Row(1, 'Bob', 25, 'SPb'),
    Row(2, 'Clare', 17, 'Moscow'),
    Row(3, 'Dan', 31, 'Kazan')
]

def RECORDREADER():
  for row in input_collection:
    yield (row.id, row)

def MAP(_, t):
  if t.age >= 18:
    yield (t, t)

def REDUCE(t, values):
  for value in values:
    yield value

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[Row(id=0, name='Ann', age=19, city='Moscow'),
 Row(id=1, name='Bob', age=25, city='SPb'),
 Row(id=3, name='Dan', age=31, city='Kazan')]

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [ ]:
class Row(NamedTuple):
  id: int
  name: str
  age: int
  city: str

input_collection = [
    Row(0, 'Ann', 19, 'Moscow'),
    Row(1, 'Bob', 25, 'SPb'),
    Row(2, 'Clare', 17, 'Moscow'),
    Row(3, 'Dan', 31, 'Kazan')
]

def RECORDREADER():
  for row in input_collection:
    yield (row.id, row)

def MAP(_, t):
  projected = (t.city,)
  yield (projected, projected)

def REDUCE(key, values):
  yield key

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('Moscow',), ('SPb',), ('Kazan',)]

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [ ]:
class Row(NamedTuple):
  id: int
  name: str

R = [
    Row(0, 'Ann'),
    Row(1, 'Bob'),
    Row(2, 'Clare')
]

S = [
    Row(2, 'Clare'),
    Row(3, 'Dan')
]

def RECORDREADER():
  for row in R:
    yield (row.id, row)
  for row in S:
    yield (row.id, row)

def MAP(_, t):
  yield (t, t)

def REDUCE(key, values):
  yield key

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[Row(id=0, name='Ann'),
 Row(id=1, name='Bob'),
 Row(id=2, name='Clare'),
 Row(id=3, name='Dan')]

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [ ]:
class Row(NamedTuple):
  id: int
  name: str

R = [
    Row(0, 'Ann'),
    Row(1, 'Bob'),
    Row(2, 'Clare')
]

S = [
    Row(2, 'Clare'),
    Row(3, 'Dan')
]

def RECORDREADER():
  for row in R:
    yield ('R', row)
  for row in S:
    yield ('S', row)

def MAP(source, t):
  yield (t, source)

def REDUCE(key, values):
  sources = set(values)
  if 'R' in sources and 'S' in sources:
    yield key

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[Row(id=2, name='Clare')]

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [ ]:
def MAP(source, t):
  yield (t, source)

def REDUCE(key, values):
  sources = set(values)
  if 'R' in sources and 'S' not in sources:
    yield key

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[Row(id=0, name='Ann'), Row(id=1, name='Bob')]

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [ ]:
class RRow(NamedTuple):
  id: int
  name: str

class SRow(NamedTuple):
  id: int
  city: str

R = [
    RRow(0, 'Ann'),
    RRow(1, 'Bob'),
    RRow(2, 'Clare')
]

S = [
    SRow(1, 'SPb'),
    SRow(2, 'Moscow'),
    SRow(3, 'Kazan')
]

def RECORDREADER():
  for row in R:
    yield ('R', row)
  for row in S:
    yield ('S', row)

def MAP(source, t):
  yield (t.id, (source, t))

def REDUCE(key, values):
  r_rows = []
  s_rows = []

  for source, t in values:
    if source == 'R':
      r_rows.append(t)
    else:
      s_rows.append(t)

  for r in r_rows:
    for s in s_rows:
      yield (r.id, r.name, s.city)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(1, 'Bob', 'SPb'), (2, 'Clare', 'Moscow')]

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [ ]:
class Row(NamedTuple):
  id: int
  name: str
  age: int
  city: str

input_collection = [
    Row(0, 'Ann', 19, 'Moscow'),
    Row(1, 'Bob', 25, 'SPb'),
    Row(2, 'Clare', 17, 'Moscow'),
    Row(3, 'Dan', 31, 'Kazan')
]

def RECORDREADER():
  for row in input_collection:
    yield (row.id, row)
def MAP(_, t):
  yield (t.city, 1)

def REDUCE(key, values):
  count = 0
  for v in values:
    count += v
  yield (key, count)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('Moscow', 2), ('SPb', 1), ('Kazan', 1)]

#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [ ]:
I = 3
J = 4

mat = np.array([
    [1., 2., 0., 3.],
    [0., 1., 4., 0.],
    [2., 0., 1., 5.]
])

vec = np.array([10., 20., 30., 40.])

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ('M', (i, j), mat[i, j])
  for j in range(vec.shape[0]):
    yield ('V', j, vec[j])

def MAP1(source, k1, v1):
  if source == 'M':
    (i, j) = k1
    yield (j, ('M', i, v1))
  else:
    j = k1
    yield (j, ('V', v1))

def REDUCE1(key, values):
  matrix_values = []
  vector_value = None

  for value in values:
    if value[0] == 'M':
      _, i, mij = value
      matrix_values.append((i, mij))
    else:
      _, vj = value
      vector_value = vj

  if vector_value is not None:
    for i, mij in matrix_values:
      yield (i, mij * vector_value)

def RECORDREADER2():
  for i, partial_value in MapReduce(RECORDREADER, MAP1, REDUCE1):
    yield (i, partial_value)

def MAP2(k1, v1):
  yield (k1, v1)

def REDUCE2(key, values):
  total = 0
  for value in values:
    total += value
  yield (key, total)

output = MapReduce(RECORDREADER2, MAP2, REDUCE2)
output = list(output)
output

reference_solution = np.matmul(mat, vec)
reference_solution

array([170., 140., 250.])

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [ ]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [ ]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):
    yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
  (i, k) = key
  s = 0
  for value in values:
    s += value
  yield ((i, k), s)

Проверьте своё решение

In [ ]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [ ]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [ ]:
import numpy as np

I = 2
J = 3
K = 4

mat_a = np.random.rand(I,J)
mat_b = np.random.rand(J,K)

def RECORDREADER():
  for i in range(mat_a.shape[0]):
    for j in range(mat_a.shape[1]):
      yield ('A', (i,j), mat_a[i,j])
  for j in range(mat_b.shape[0]):
    for k in range(mat_b.shape[1]):
      yield ('B', (j,k), mat_b[j,k])

def MAP(source, k1, v1):
  if source == 'A':
    (i, j) = k1
    v = v1
    yield (j, ('A', i, v))
  else:
    (j, k) = k1
    w = v1
    yield (j, ('B', k, w))

def REDUCE(key, values):
  a_values = []
  b_values = []

  for value in values:
    if value[0] == 'A':
      _, i, v = value
      a_values.append((i, v))
    else:
      _, k, w = value
      b_values.append((k, w))

  for i, v in a_values:
    for k, w in b_values:
      yield ((i, k), v * w)

def RECORDREADER2():
  for key, value in MapReduce(RECORDREADER, MAP, REDUCE):
    yield (key, value)

def MAP2(k1, v1):
  yield (k1, v1)

def REDUCE2(key, values):
  s = 0
  for value in values:
    s += value
  yield (key, s)


# CHECK THE SOLUTION
reference_solution = np.matmul(mat_a, mat_b)
solution = MapReduce(RECORDREADER2, MAP2, REDUCE2)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [ ]:
I = 2
J = 3
K = 4

mat_a = np.random.rand(I,J)
mat_b = np.random.rand(J,K)

maps = 2
reducers = 2

def INPUTFORMAT():
  def RECORDREADER_A():
    for i in range(mat_a.shape[0]):
      for j in range(mat_a.shape[1]):
        yield (('A', i, j), mat_a[i,j])

  def RECORDREADER_B():
    for j in range(mat_b.shape[0]):
      for k in range(mat_b.shape[1]):
        yield (('B', j, k), mat_b[j,k])

  yield RECORDREADER_A()
  yield RECORDREADER_B()

def MAP(k1, v1):
  source, x, y = k1

  if source == 'A':
    i, j = x, y
    v = v1
    yield (j, ('A', i, v))
  else:
    j, k = x, y
    w = v1
    yield (j, ('B', k, w))

def REDUCE(j, values):
  a_values = []
  b_values = []

  for value in values:
    if value[0] == 'A':
      _, i, v = value
      a_values.append((i, v))
    else:
      _, k, w = value
      b_values.append((k, w))

  for i, v in a_values:
    for k, w in b_values:
      yield ((i, k), v*w)

  # CHECK THE SOLUTION
reference_solution = np.matmul(mat_a, mat_b)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]

def RECORDREADER2():
  for partition_id, partition in partitioned_output:
    for item in partition:
      if item is not None:
        yield item

def MAP2(k1, v1):
  yield (k1, v1)

def REDUCE2(key, values):
  s = 0
  for value in values:
    s += value
  yield (key, s)

solution = MapReduce(RECORDREADER2, MAP2, REDUCE2)

def asmatrix(reduce_output):
  mat = np.zeros((I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

18 key-value pairs were sent over a network.


True

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [ ]:
I = 2
J = 3
K = 4

mat_a = np.random.rand(I,J)
mat_b = np.random.rand(J,K)

maps = 4
reducers = 2

def INPUTFORMAT():
  def RECORDREADER_A_1():
    for i in range(0, mat_a.shape[0]):
      for j in range(0, 1):
        yield (('A', i, j), mat_a[i,j])

  def RECORDREADER_A_2():
    for i in range(0, mat_a.shape[0]):
      for j in range(1, mat_a.shape[1]):
        yield (('A', i, j), mat_a[i,j])

  def RECORDREADER_B_1():
    for j in range(0, 1):
      for k in range(0, mat_b.shape[1]):
        yield (('B', j, k), mat_b[j,k])

  def RECORDREADER_B_2():
    for j in range(1, mat_b.shape[0]):
      for k in range(0, mat_b.shape[1]):
        yield (('B', j, k), mat_b[j,k])

  yield RECORDREADER_A_1()
  yield RECORDREADER_A_2()
  yield RECORDREADER_B_1()
  yield RECORDREADER_B_2()

def MAP(k1, v1):
  source, x, y = k1

  if source == 'A':
    i, j = x, y
    v = v1
    yield (j, ('A', i, v))
  else:
    j, k = x, y
    w = v1
    yield (j, ('B', k, w))

def REDUCE(j, values):
  a_values = []
  b_values = []

  for value in values:
    if value[0] == 'A':
      _, i, v = value
      a_values.append((i, v))
    else:
      _, k, w = value
      b_values.append((k, w))

  for i, v in a_values:
    for k, w in b_values:
      yield ((i, k), v*w)

  # CHECK THE SOLUTION
reference_solution = np.matmul(mat_a, mat_b)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]

def RECORDREADER2():
  for partition_id, partition in partitioned_output:
    for item in partition:
      if item is not None:
        yield item

def MAP2(k1, v1):
  yield (k1, v1)

def REDUCE2(key, values):
  s = 0
  for value in values:
    s += value
  yield (key, s)

solution = MapReduce(RECORDREADER2, MAP2, REDUCE2)

def asmatrix(reduce_output):
  mat = np.zeros((I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

18 key-value pairs were sent over a network.


True

"Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?"

Да, будет, если в совокупности все RECORDREADER`ы порождают каждый ненулевой элемент матриц ровно один раз.

Если какие-то элементы будут потеряны, результат будет неверным.
Если одни и те же элементы будут сгенерированы несколько раз, их вклад посчитается несколько раз и результат тоже будет неверным.

То есть случайные разбиение допустимо, а случайная неполная или пересекающаяся выборка нет.